# Experiment 09: hard-selection and loss ablation

This experiment tests why targeted fine-tuning may not outperform random training. It compares three selection rules using out-of-fold predictions: uncertainty, incorrect-and-uncertain, and highest true-label loss. Every targeted condition has a class-matched random control with the same number of selected examples. All continuation runs start from the same baseline checkpoint, use the same optimizer and epochs, and use an unweighted validation macro-F1 for checkpoint selection.

## Imports and configuration

In [1]:
import copy
import os
import random
import sys

os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

sys.path.append("..")
from utils.preprocessing import LABELS, encode_labels
from utils.hard_samples_identify import get_hard_sample_indices

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 64
BATCH_SIZE = 16
BASELINE_EPOCHS = 3
SELECTION_EPOCHS = 1
TARGETED_EPOCHS = 2
LEARNING_RATE = 2e-5
HARD_PERCENTILE = 20
HARD_WEIGHT = 1.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Device:", DEVICE)


Device: cpu


## Load and tokenize data

In [2]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
y_test = encode_labels(test_df.label)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(frame):
    values = tokenizer(frame.sentence.tolist(), padding="max_length", truncation=True,
                       max_length=MAX_LENGTH, return_tensors="pt")
    return values["input_ids"], values["attention_mask"]

input_train, mask_train = tokenize(train_df)
input_val, mask_val = tokenize(val_df)
input_test, mask_test = tokenize(test_df)


## Transformer training helpers

In [3]:
def loader(input_ids, masks, labels, weights=None, shuffle=False):
    tensors = [input_ids, masks, torch.tensor(labels, dtype=torch.long)]
    if weights is not None: tensors.append(torch.tensor(weights, dtype=torch.float32))
    return DataLoader(TensorDataset(*tensors), batch_size=BATCH_SIZE, shuffle=shuffle)

def new_model(seed=SEED):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)),
        label2id={label: i for i, label in enumerate(LABELS)},
    )
    return model.to(DEVICE)

@torch.no_grad()
def predict(model, input_ids, masks, labels):
    model.eval(); probabilities=[]; predictions=[]; losses=[]
    for batch in loader(input_ids, masks, labels):
        ids, batch_masks, batch_labels = [value.to(DEVICE) for value in batch]
        output = model(input_ids=ids, attention_mask=batch_masks, labels=batch_labels)
        probabilities.append(torch.softmax(output.logits, dim=1).cpu().numpy())
        predictions.append(output.logits.argmax(dim=1).cpu().numpy())
        losses.append(torch.nn.functional.cross_entropy(output.logits, batch_labels, reduction="none").cpu().numpy())
    return np.concatenate(probabilities), np.concatenate(predictions), np.concatenate(losses)

def fit(model, input_ids, masks, labels, epochs, weights=None):
    train_loader = loader(input_ids, masks, labels, weights, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, max(1, len(train_loader)//10), max(1, len(train_loader)*epochs)
    )
    best_state = copy.deepcopy(model.state_dict()); best_f1 = -np.inf
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            ids, batch_masks, batch_labels, *extra = [value.to(DEVICE) for value in batch]
            output = model(input_ids=ids, attention_mask=batch_masks, labels=batch_labels)
            if extra:
                per_item = torch.nn.functional.cross_entropy(output.logits, batch_labels, reduction="none")
                loss = (per_item * extra[0]).mean()
            else:
                loss = output.loss
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
        val_probs, val_preds, _ = predict(model, input_val, mask_val, y_val)
        val_f1 = f1_score(y_val, val_preds, average="macro")
        print(f"epoch {epoch + 1}/{epochs} - val_macro_f1: {val_f1:.4f}")
        if val_f1 > best_f1: best_f1, best_state = val_f1, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model

def score(model, input_ids, masks, labels, hard_indices):
    probabilities, predictions, _ = predict(model, input_ids, masks, labels)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "hard_accuracy": accuracy_score(labels[hard_indices], predictions[hard_indices]),
        "hard_macro_f1": f1_score(labels[hard_indices], predictions[hard_indices], average="macro"),
    }, probabilities, predictions


## Train baseline and produce out-of-fold selection predictions

In [4]:
baseline_model = fit(new_model(SEED), input_train, mask_train, y_train, BASELINE_EPOCHS)
baseline_test_probs, _, _ = predict(baseline_model, input_test, mask_test, y_test)

oof_probs = np.zeros((len(y_train), len(LABELS)))
oof_losses = np.zeros(len(y_train))
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
for fold, (fit_idx, held_idx) in enumerate(skf.split(input_train, y_train)):
    fold_model = fit(new_model(SEED + fold + 1), input_train[fit_idx], mask_train[fit_idx], y_train[fit_idx], SELECTION_EPOCHS)
    probs, _, losses = predict(fold_model, input_train[held_idx], mask_train[held_idx], y_train[held_idx])
    oof_probs[held_idx] = probs; oof_losses[held_idx] = losses
    del fold_model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()


W1027 13:36:15.718000 99595 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/3 - val_macro_f1: 0.8017
epoch 2/3 - val_macro_f1: 0.8195
epoch 3/3 - val_macro_f1: 0.8119


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/1 - val_macro_f1: 0.4966


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/1 - val_macro_f1: 0.5622


## Build and diagnose selection sets

In [5]:
def margins(probabilities):
    top_two = np.partition(probabilities, -2, axis=1)[:, -2:]
    return top_two.max(axis=1) - top_two.min(axis=1)

oof_predictions = oof_probs.argmax(axis=1)
oof_margins = margins(oof_probs)
uncertain_indices, uncertain_cutoff = get_hard_sample_indices(oof_probs, HARD_PERCENTILE)
loss_cutoff = np.percentile(oof_losses, 100 - HARD_PERCENTILE)
error_uncertain_indices = np.flatnonzero((oof_predictions != y_train) & (oof_margins <= uncertain_cutoff))
loss_indices = np.argsort(oof_losses)[-len(uncertain_indices):]
selection_sets = {
    "uncertainty": uncertain_indices,
    "error_uncertainty": error_uncertain_indices,
    "true_label_loss": loss_indices,
}

def class_counts(indices): return np.bincount(y_train[indices], minlength=len(LABELS)).tolist()
for name, indices in selection_sets.items():
    print(name, {
        "count": len(indices),
        "error_rate": float(np.mean(oof_predictions[indices] != y_train[indices])),
        "mean_oof_loss": float(np.mean(oof_losses[indices])),
        "mean_margin": float(np.mean(oof_margins[indices])),
        "class_counts": class_counts(indices),
    })
print("uncertainty cutoff:", uncertain_cutoff, "loss cutoff:", loss_cutoff)


uncertainty {'count': 580, 'error_rate': 0.5568965517241379, 'mean_oof_loss': 0.9598970151153104, 'mean_margin': 0.06873316996056458, 'class_counts': [231, 128, 221]}
error_uncertainty {'count': 323, 'error_rate': 1.0, 'mean_oof_loss': 1.083789699402387, 'mean_margin': 0.07021460896675062, 'class_counts': [187, 62, 74]}
true_label_loss {'count': 580, 'error_rate': 1.0, 'mean_oof_loss': 1.5245236115208987, 'mean_margin': 0.3478774801410478, 'class_counts': [214, 48, 318]}
uncertainty cutoff: 0.132053852081 loss cutoff: 1.0873997211456299


## Class-matched random controls and targeted continuation

In [6]:
def matched_random_indices(selected_indices, seed):
    rng = np.random.default_rng(seed)
    result=[]
    selected_labels = y_train[selected_indices]
    for class_id in range(len(LABELS)):
        needed = int(np.sum(selected_labels == class_id))
        available = np.flatnonzero(y_train == class_id)
        result.extend(rng.choice(available, size=needed, replace=False))
    return np.asarray(result, dtype=int)

test_hard_indices, test_cutoff = get_hard_sample_indices(
    baseline_test_probs, HARD_PERCENTILE
)

results = []
baseline_scores, _, _ = score(baseline_model, input_test, mask_test, y_test, test_hard_indices)
results.append({"strategy": "baseline", "selected_count": 0, **baseline_scores})

for run_number, (selection_name, selected_indices) in enumerate(selection_sets.items()):
    random_indices = matched_random_indices(selected_indices, SEED + 100 + run_number)
    for strategy, indices in [
        (selection_name, selected_indices),
        (f"{selection_name}_random_control", random_indices),
    ]:
        weights = np.ones(len(y_train), dtype=np.float32)
        weights[indices] = HARD_WEIGHT
        model = new_model(SEED + 200 + run_number * 2 + int(strategy.endswith("random_control")))
        model.load_state_dict(copy.deepcopy(baseline_model.state_dict()))
        model = fit(model, input_train, mask_train, y_train, TARGETED_EPOCHS, weights=weights)
        scores, _, _ = score(model, input_test, mask_test, y_test, test_hard_indices)
        results.append({"strategy": strategy, "selected_count": len(indices), **scores})
        del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/2 - val_macro_f1: 0.8139
epoch 2/2 - val_macro_f1: 0.8280


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/2 - val_macro_f1: 0.8258
epoch 2/2 - val_macro_f1: 0.8218


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/2 - val_macro_f1: 0.8120
epoch 2/2 - val_macro_f1: 0.8213


## Evaluate selection and training effects

In [7]:
results_df = pd.DataFrame(results)
for column in ["accuracy", "macro_f1", "hard_accuracy", "hard_macro_f1"]:
    results_df[f"delta_{column}_vs_baseline"] = results_df[column] - results_df.loc[0, column]
print("Test hard subset size:", len(test_hard_indices))
print(results_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))


Test hard subset size: 194
                        strategy  selected_count  accuracy  macro_f1  hard_accuracy  hard_macro_f1  delta_accuracy_vs_baseline  delta_macro_f1_vs_baseline  delta_hard_accuracy_vs_baseline  delta_hard_macro_f1_vs_baseline
                        baseline               0    0.8523    0.8340         0.6186         0.6258                      0.0000                      0.0000                           0.0000                           0.0000
                     uncertainty             580    0.8502    0.8360         0.5979         0.6072                     -0.0021                      0.0020                          -0.0206                          -0.0186
      uncertainty_random_control             580    0.8543    0.8405         0.6237         0.6410                      0.0021                      0.0065                           0.0052                           0.0152
               error_uncertainty             323    0.8512    0.8373         0.6031      

## Save results

In [8]:
# os.makedirs("../models", exist_ok=True)
# results_df.to_csv("../models/experiment_09_selection_loss_ablation.csv", index=False)
# tokenizer.save_pretrained("../models/experiment_09_distilbert_tokenizer")
